# UD5 - Dashboard en Tiempo Real con Dash y CoinGecko

En el notebook anterior construimos un dashboard con datos **estáticos** (los cargamos una vez y no cambian).

Ahora vamos un paso más allá: vamos a conectarnos a una **API real** que devuelve precios de criptomonedas actualizados, y haremos que el dashboard se **refresque solo** cada cierto tiempo sin que el usuario tenga que hacer nada.

---
## Instalación e imports

In [1]:
!pip install dash requests --quiet

In [2]:
import requests
import json
from datetime import datetime
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from dash import Dash, html, dcc, callback, Output, Input, State

---
## 1. La API antes de tocar Dash

Vamos a usar la API pública de **CoinGecko** — gratuita, sin registro, sin clave.

El endpoint que nos interesa es:
```
https://api.coingecko.com/api/v3/simple/price
```

Acepta parámetros en la URL:
- `ids` → qué monedas queremos (separadas por coma)
- `vs_currencies` → en qué divisa queremos el precio
- `include_24hr_change` → si queremos la variación de las últimas 24h

In [3]:
# Definimos la URL y los parámetros por separado (más limpio que todo en la URL)
URL = "https://api.coingecko.com/api/v3/simple/price"

PARAMS = {
    "ids": "bitcoin,ethereum,solana",
    "vs_currencies": "eur",
    "include_24hr_change": "true",
}

respuesta = requests.get(URL, params=PARAMS)

# Comprobamos que ha ido bien (200 = OK)
print(f"Código de respuesta: {respuesta.status_code}")

datos = respuesta.json()
print(json.dumps(datos, indent=2))

Código de respuesta: 200
{
  "bitcoin": {
    "eur": 65462,
    "eur_24h_change": 1.0508585571921587
  },
  "ethereum": {
    "eur": 1966.38,
    "eur_24h_change": 1.3997752864096025
  },
  "solana": {
    "eur": 71.74,
    "eur_24h_change": 0.9473481678327196
  }
}


In [4]:
# Accedemos a los datos de cada moneda
print(f"Bitcoin:  {datos['bitcoin']['eur']:,.0f} € ({datos['bitcoin']['eur_24h_change']:+.2f}%)")
print(f"Ethereum: {datos['ethereum']['eur']:,.0f} € ({datos['ethereum']['eur_24h_change']:+.2f}%)")
print(f"Solana:   {datos['solana']['eur']:,.0f} € ({datos['solana']['eur_24h_change']:+.2f}%)")

Bitcoin:  65,462 € (+1.05%)
Ethereum: 1,966 € (+1.40%)
Solana:   72 € (+0.95%)


In [5]:
MONEDAS = {
    "bitcoin":  "Bitcoin",
    "ethereum": "Ethereum",
    "solana":   "Solana",
}

def obtener_precios():
    """Llama a la API y devuelve un diccionario limpio con precio y cambio 24h.
    Si la API falla (sin internet, límite de peticiones - 30 peticiones por minuto), devuelve None.
    """
    try:
        r = requests.get(URL, params=PARAMS, timeout=5)
        r.raise_for_status()   # Lanza error si status != 200
        raw = r.json()
        
        resultado = {}
        for clave, nombre in MONEDAS.items():
            resultado[nombre] = {
                "precio": raw[clave]["eur"],
                "cambio": raw[clave]["eur_24h_change"],
            }
        resultado["timestamp"] = datetime.now().strftime("%H:%M:%S")
        return resultado
    
    except Exception as e:
        print(f"Error al llamar a la API: {e}")
        return None

# Probamos la función
precios = obtener_precios()
print(precios)

{'Bitcoin': {'precio': 65462, 'cambio': 1.0508585571921587}, 'Ethereum': {'precio': 1966.38, 'cambio': 1.3997752864096025}, 'Solana': {'precio': 71.74, 'cambio': 0.9473481678327196}, 'timestamp': '16:48:12'}


---
## 2. Layout estático con esos datos

Construimos el dashboard con lo que ya sabemos. Por ahora, los datos se cargan **una sola vez** al arrancar la app — exactamente igual que con StreamFlow.

In [6]:
# =============================================
# ESTILOS (los mismos que ya conocemos)
# =============================================

estilo_pagina = {'fontFamily': 'Arial, sans-serif', 'maxWidth': '1100px',
                 'margin': '0 auto', 'padding': '20px', 'backgroundColor': '#0d1117', 'minHeight': '100vh'}

estilo_titulo = {'textAlign': 'center', 'color': '#f0f6fc', 'marginBottom': '4px'}
estilo_subtitulo = {'textAlign': 'center', 'color': '#8b949e', 'marginTop': '0', 'fontSize': '14px'}

estilo_fila = {'display': 'flex', 'marginBottom': '20px', 'gap': '16px'}
estilo_columna = {'flex': '1'}

def estilo_tarjeta(color_acento='#58a6ff'):
    return {
        'backgroundColor': '#161b22',
        'borderRadius': '10px',
        'padding': '20px',
        'textAlign': 'center',
        'flex': '1',
        'borderTop': f'3px solid {color_acento}',
    }

estilo_nombre_moneda = {'fontSize': '13px', 'color': '#8b949e', 'margin': '0 0 6px 0'}
estilo_precio = {'fontSize': '32px', 'fontWeight': 'bold', 'color': '#f0f6fc', 'margin': '0'}
estilo_cambio = {'fontSize': '14px', 'margin': '6px 0 0 0'}

# Color de la variación según si sube o baja
def color_cambio(cambio):
    return 'green' if cambio >= 0 else 'red'

def flecha(cambio):
    return '▲' if cambio >= 0 else '▼'

In [7]:
app = Dash(__name__)

precios = obtener_precios()
nombres= ['Bitcoin','Ethereum','Solana']
colores = ['orange','royalblue','purple']
precios_lista = [precios[m]['precio'] for m in nombres]

fig_barras = px.bar(
    x=nombres,
    y=precios_lista,
    color=nombres,
    color_discrete_sequence=colores,
    labels={'x': 'Moneda', 'y':'Precio (€)'},
    title='Precio actual en €',
    template='plotly_dark',
)
fig_barras.update_layout(showlegend=False) #Quitar la leyenda

app.layout = html.Div([
    html.H1('Dashboard Criptomonedas', style = estilo_titulo),
    html.P(f'Última actualización: {precios["timestamp"]}', style = estilo_subtitulo),
    html.Hr(),

    html.Div([
        html.Div([
            html.P('Bitcoin',style=estilo_nombre_moneda),
            html.P(f"{precios['Bitcoin']['precio']:,.0f} €",style=estilo_precio),
            html.P(f"{flecha(precios['Bitcoin']['cambio'])} {precios['Bitcoin']['cambio']:+.2f}% (24h)",style={'color': color_cambio(precios['Bitcoin']['cambio']), 'fontSize': '14px', 'margin': '6px 0 0 0'}),
        ],style=estilo_tarjeta('orange')),
        html.Div([
            html.P('Ethereum',style=estilo_nombre_moneda),
            html.P(f"{precios['Ethereum']['precio']:,.0f} €",style=estilo_precio),
            html.P(f"{flecha(precios['Ethereum']['cambio'])} {precios['Ethereum']['cambio']:+.2f}% (24h)", style={'color': color_cambio(precios['Bitcoin']['cambio']), 'fontSize': '14px', 'margin': '6px 0 0 0'}),
        ],style=estilo_tarjeta('royalblue')),
        html.Div([
            html.P('Solana',style=estilo_nombre_moneda),
            html.P(f"{precios['Solana']['precio']:,.0f} €",style=estilo_precio),
            html.P(f"{flecha(precios['Solana']['cambio'])} {precios['Solana']['cambio']:+.2f}% (24h)",style={'color': color_cambio(precios['Bitcoin']['cambio']), 'fontSize': '14px', 'margin': '6px 0 0 0'}),
        ],style=estilo_tarjeta('purple')),
    ],style= estilo_fila),
    
    dcc.Graph(figure=fig_barras),
    
], style=estilo_pagina)

app.run(host='0.0.0.0',port=8053, jupyter_mode='external')
print("http://localhost:8053")

    

Dash app running on http://0.0.0.0:8053/
http://localhost:8053


TIEMPO REAL -> dcc.Interval

In [8]:
app = Dash(__name__)

app.layout = html.Div([
    #Definir este intervalo que será como un reloj invisible 
    dcc.Interval(
        id='intervalo',
        interval=120000, #en milisengundos
        n_intervals=0
    ),
    html.H1('Dashboard Criptomonedas - Tiempo Real', style = estilo_titulo),
    html.P(f'Última actualización: {precios["timestamp"]}', style = estilo_subtitulo),
    html.Hr(),
    
    #Las KPIs y la Gráfica
    html.Div([
        html.Div([
            html.P('Bitcoin',style=estilo_nombre_moneda),
            html.P(f"{precios['Bitcoin']['precio']:,.0f} €",style=estilo_precio),
            html.P(f"{flecha(precios['Bitcoin']['cambio'])} {precios['Bitcoin']['cambio']:+.2f}% (24h)",style={'color': color_cambio(precios['Bitcoin']['cambio']), 'fontSize': '14px', 'margin': '6px 0 0 0'}),
        ],style=estilo_tarjeta('orange')),
        html.Div([
            html.P('Ethereum',style=estilo_nombre_moneda),
            html.P(f"{precios['Ethereum']['precio']:,.0f} €",style=estilo_precio),
            html.P(f"{flecha(precios['Ethereum']['cambio'])} {precios['Ethereum']['cambio']:+.2f}% (24h)", style={'color': color_cambio(precios['Bitcoin']['cambio']), 'fontSize': '14px', 'margin': '6px 0 0 0'}),
        ],style=estilo_tarjeta('royalblue')),
        html.Div([
            html.P('Solana',style=estilo_nombre_moneda),
            html.P(f"{precios['Solana']['precio']:,.0f} €",style=estilo_precio),
            html.P(f"{flecha(precios['Solana']['cambio'])} {precios['Solana']['cambio']:+.2f}% (24h)",style={'color': color_cambio(precios['Bitcoin']['cambio']), 'fontSize': '14px', 'margin': '6px 0 0 0'}),
        ],style=estilo_tarjeta('purple')),
        dcc.Graph(figure=fig_barras),

    ],style= estilo_fila),
    
    dcc.Graph(id='grafico-barras'),
    
], style=estilo_pagina)

#Las propiedades de un componente html.P
#html.P(children='Bitcoin')
            #style
            #disable
#CALLBACKS
@callback(
    Output('ultima_actualizacion','children'),
    Output('bitcoin-precio','children'),
    Output('ethereum-precio','children'),
    Output('solana-precio','children'),
    Output('grafico-barras','figure'),

    Input('intervalo','n_intervals'),
)
def actualizar_todo(n_intervals):

    precios = obtener_precios()
    subtitulos = precios['timestamp']

    bitcoin_p = f"{precios['Bitcoin']['precio']}"
    ethereum_p = f"{precios['Ethereum']['precio']}"
    bitcoin_p = f"{precios['Bitcoin']['precio']}"
    
    #Completamos las KPIs

    #Gráfica
    nombres= ['Bitcoin','Ethereum','Solana']
    colores = ['orange','royalblue','purple']
    precios_lista = [precios[m]['precio'] for m in nombres]
    
    fig_barras = px.bar(
        x=nombres,
        y=precios_lista,
        color=nombres,
        color_discrete_sequence=colores,
        labels={'x': 'Moneda', 'y':'Precio (€)'},
        title='Precio actual en €',
        template='plotly_dark',
    )

            
    return(subtitulo, bitoin_p, solana_p, ethereum_p,fig_barras)

app.run(host='0.0.0.0',port=8054, jupyter_mode='external')
print("http://localhost:8054")

Dash app running on http://0.0.0.0:8054/
http://localhost:8054


In [ ]:
# Se pueden pasar datos sin disparar el Callback
#State('historial-data', 'data')


df = pd.DataFrame(historial)
fig_lineas = px.line(
    df,
    x = 'tiempo',
    y = ['Bitcoin', 'Ethereum', 'Solana']
)